# Generate a PhaseGridUNet dataset

This notebook calls reusable source modules. Check the saved ACTION before Run All; use none to skip generation. Use the grid-geometry uv environment and an available matplotlib installation; no asset downloads are needed for the procedural scenes. An existing destination is never overwritten.


In [ ]:
import json
import platform
import sys
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
from grid_geometry.phase.dataset import PhaseDataset, inspect_dataset
from grid_geometry.phase.generate_dataset import GenerationConfig, generate_dataset, sample_fractions
from grid_geometry.phase.targets import PhaseGeometry

print(sys.version, platform.platform(), "torch", torch.__version__)
PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
print("Project:", PROJECT.resolve())


## Configuration
Choose an action deliberately. Counts are provisional, not a training budget. For inspection only, set ACTION to `inspect` and point DESTINATION at an existing completed dataset.


In [ ]:
ACTION = "none"  # none, smoke, sampling, development, inspect
WORKERS = 8
SEED = 42
PNG_COMPRESSION = 1
INPUT_SIZE = (384, 384)
TRANSITION_RATIO = .25
PADDING_EXAMPLE_FRACTION = .05
PADDING_MAX_FRACTION = .20
CONTEXT_WEIGHTS = {"procedural": 70, "desktop": 30}
COVERAGE_EXPLORATION_FRACTION = .12
COVERAGE_PREFERENCE_EXPONENT = 1.5
DESKTOP_FRACTION = .8

PITCH_BINS = (
    (6., 11., .10),
    (12., 19., .25),
    (20., 31., .35),
    (32., 47., .20),
    (48., 64., .10),
)

CATEGORIES = {
    "complete": .35,
    "interior": .25,
    "side": .20,
    "corner": .20,
}

PRESETS = {
    "smoke": (40, 12, 12),
    "sampling": (500, 0, 0),
    "development": (20000, 2000, 2000),
}

DESTINATION = PROJECT / "datasets" / "phase_training_smoke"

print("Action:", ACTION, "Destination:", DESTINATION)

## Generation
The generator records an incomplete marker until all splits are written. Choose a fresh destination for every new run.


In [ ]:
if ACTION in PRESETS:
    train, val, test = PRESETS[ACTION]
    config = GenerationConfig(train=train, val=val, test=test, seed=SEED, workers=WORKERS,
        png_compression=PNG_COMPRESSION, input_size=INPUT_SIZE, pitch_bins=PITCH_BINS,
        categories=CATEGORIES, transition_ratio=TRANSITION_RATIO,
        padding_example_fraction=PADDING_EXAMPLE_FRACTION, padding_max_fraction=PADDING_MAX_FRACTION,
        context_weights=CONTEXT_WEIGHTS,
        coverage_exploration_fraction=COVERAGE_EXPLORATION_FRACTION,
        coverage_preference_exponent=COVERAGE_PREFERENCE_EXPONENT, desktop_fraction=DESKTOP_FRACTION)
    statistics = generate_dataset(DESTINATION, config)
    print(json.dumps(statistics, indent=2))
elif ACTION not in ("none", "inspect"):
    raise ValueError("Unknown action")


## Integrity and distributions
Integrity inspection streams all samples and targets; it can take time on a development dataset. Empty splits are supported.


In [ ]:
if ACTION != "none":
    print(json.dumps(inspect_dataset(DESTINATION), indent=2))
    stats = json.loads((DESTINATION / "statistics.json").read_text())
    print(json.dumps(stats, indent=2))
    dataset_config = json.loads((DESTINATION / "config.json").read_text())
    records = [json.loads(line) for line in (DESTINATION / "manifest.jsonl").read_text().splitlines()]
    # Reconstruct the compact metrics when inspecting an older dataset.
    for r in records:
        geometry = PhaseGeometry(**r["geometry"]) if r["geometry"] else None
        r.update(sample_fractions(tuple(dataset_config["input_size"]), r["known_box"], geometry,
                                  dataset_config["transition_ratio"]))
    positives = [r for r in records if r["board_id"] is not None]
    fig, ax = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
    ax[0, 0].hist([r["effective_pitch_x"] for r in positives], bins=20, alpha=.6, label="x")
    ax[0, 0].hist([r["effective_pitch_y"] for r in positives], bins=20, alpha=.6, label="y")
    ax[0, 0].set_title("Realized effective pitch"); ax[0, 0].legend()
    categories = Counter(r["category"] for r in records)
    ax[0, 1].bar(categories.keys(), categories.values()); ax[0, 1].tick_params(axis="x", rotation=35)
    ax[0, 1].set_title("Sampling categories")
    ax[1, 0].hist([r["grid_coverage_fraction"] for r in positives], bins=20, range=(0, 1))
    ax[1, 0].set_title("Positive grid coverage")
    ax[1, 1].hist([r["padding_fraction"] for r in records], bins=20, range=(0, 1))
    ax[1, 1].set_title("Artificial padding fraction"); plt.show()


## Representative samples and target fields
Selection includes categories, styles, and pitch extremes when present. Signed fields use a fixed diverging range; masked phase is transparent. The lattice overlay is the ideal lattice, not necessarily the visible Microsoft tile extent.


In [ ]:
if ACTION != "none":
    selected = {}
    for r in records:
        keys = [r["category"]]
        if r["style"]: keys.append(r["style"])
        if r.get("context_type"): keys.append("context: " + r["context_type"])
        if r.get("padding_type"): keys.append("padded " + ("positive" if r["board_id"] else "negative"))
        for key in keys: selected.setdefault(key, r)
    if positives:
        selected["small pitch"] = min(positives, key=lambda r: r["effective_pitch_x"])
        selected["large pitch"] = max(positives, key=lambda r: r["effective_pitch_x"])
        selected["lowest coverage"] = min(positives, key=lambda r: r["grid_coverage_fraction"])
    if records:
        selected["largest padding"] = max(records, key=lambda r: r["padding_fraction"])
    # Keep every selection label, but plot a given split/sample only once.
    unique = {}
    for label, r in selected.items():
        key = (r["split"], r["sample_id"])
        if key not in unique: unique[key] = ([], r)
        unique[key][0].append(label)
    for labels, r in unique.values():
        dataset = PhaseDataset(DESTINATION, r["split"])
        sample = dataset.load_sample(dataset.names.index(r["sample_id"]))
        targets = sample.targets.numpy(); mask = sample.phase_mask.numpy()
        fig, axes = plt.subplots(2, 4, figsize=(16, 9), constrained_layout=True)
        axes = axes.ravel(); axes[0].imshow(sample.image.permute(1, 2, 0), extent=(0, dataset.size[0], dataset.size[1], 0))
        axes[0].set_title(", ".join(labels) + " / RGB", fontsize=9)
        geometry = r["geometry"]
        if geometry:
            w, h = dataset.size
            ox, oy, px, py = geometry["origin_x"], geometry["origin_y"], geometry["pitch_x"], geometry["pitch_y"]
            rows, columns = r["board_dimensions"]
            end_x, end_y = ox + columns * px, oy + rows * py
            left, right = max(0, ox), min(w, end_x)
            top, bottom = max(0, oy), min(h, end_y)
            if left < right and top < bottom:
                for x in ox + np.arange(columns + 1) * px:
                    if 0 <= x <= w: axes[0].plot([x, x], [top, bottom], color="yellow", lw=.5)
                for y in oy + np.arange(rows + 1) * py:
                    if 0 <= y <= h: axes[0].plot([left, right], [y, y], color="yellow", lw=.5)
            from matplotlib.patches import Rectangle
            vl, vt, vr, vb = geometry["visible_box"]
            axes[0].add_patch(Rectangle((vl, vt), vr-vl, vb-vt, fill=False, edgecolor="cyan", lw=1.2, linestyle="--"))
            axes[0].set_xlim(0, w); axes[0].set_ylim(h, 0)
        for i, title in enumerate(("X cosine", "X sine", "Y cosine", "Y sine")):
            axes[i+1].imshow(np.ma.masked_where(~mask, targets[i]), cmap="coolwarm", vmin=-1, vmax=1)
            axes[i+1].set_title(title)
        axes[5].imshow(targets[4], cmap="gray", vmin=0, vmax=1); axes[5].set_title("Occupancy")
        axes[6].imshow(mask, cmap="gray", vmin=0, vmax=1); axes[6].set_title("Phase mask")
        axes[7].imshow(sample.occupancy_mask, cmap="gray", vmin=0, vmax=1); axes[7].set_title("Occupancy mask")
        for ax in axes: ax.axis("off")
        fig.suptitle(f"{r['split']}/{r['sample_id']}  {r['category']}  {r['style']}  pitch={r['effective_pitch_x']}  coverage={r['grid_coverage_fraction']}  padding={r['padding_fraction']:.1%}", fontsize=11)
        plt.show()
